In [21]:
from langchain.document_loaders import JSONLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.docstore.document import Document
import json

In [22]:
def load_data(json_path="recipes.json"):
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    docs = [Document(page_content=entry['content'], metadata={'title': entry['title'],'type':entry['type']}) for entry in data]
    return docs

In [23]:
documents = load_data()
print(documents)

[Document(metadata={'title': 'Dal Bhat', 'type': 'main course'}, page_content='Ingredients: 1 cup rice, 1/2 cup lentils (dal), 1/2 tsp turmeric, salt to taste, vegetables (spinach, cauliflower, etc.), tomato achar. Instructions: 1. Rinse lentils and boil with turmeric and salt until soft. 2. In another pot, cook rice until fluffy. 3. Stir-fry seasonal vegetables with spices. 4. Serve rice with dal, vegetables, and achar.'), Document(metadata={'title': 'Momo (Dumplings)', 'type': 'main course'}, page_content='Ingredients: All-purpose flour, minced chicken or vegetables, garlic, ginger, onion, soy sauce, salt to taste. Instructions: 1. Knead dough using flour and water. 2. Mix filling ingredients together. 3. Roll dough, fill with mixture, and shape into dumplings. 4. Steam for 10 to 15 minutes or deep-fry. 5. Serve with spicy tomato achar.'), Document(metadata={'title': 'Sel Roti', 'type': 'snack'}, page_content='Ingredients: 2 cups rice flour, 1 ripe banana, 1/4 cup sugar, 1/2 cup wate

In [24]:
# splitter = RecursiveCharacterTextSplitter(
#     chunk_size = 300,
#     chunk_overlap = 50
# )

# chunks = splitter.split_documents(documents)

In [25]:
# print(chunks)

In [26]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)


In [27]:
from langchain.vectorstores import Chroma
# vectorstore = Chroma.from_documents(chunks,embedding_model)
vectorstore = Chroma.from_documents(documents,embedding_model)

In [28]:
from langchain.retrievers.self_query.base import SelfQueryRetriever
from langchain.chains.query_constructor.base import AttributeInfo

metadata_field_info =[
      AttributeInfo(name="title", description="The title of the recipe", type="string"),
      AttributeInfo(name="type", description="The type of the recipe", type="string"),
]

document_content_description = "Brief summary about food recipe"

## Hugging face LLM model

In [29]:
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint

from dotenv import load_dotenv

load_dotenv()

model = HuggingFaceEndpoint(
    repo_id="meta-llama/Meta-Llama-3-8B-Instruct",
    task="text-generation"
)

llm = ChatHuggingFace(llm=model)


## Gemini api

In [30]:
# import os
# from dotenv import load_dotenv
# from langchain_google_genai import ChatGoogleGenerativeAI

# load_dotenv()

# llm = ChatGoogleGenerativeAI(model='gemini-1.5-flash',api_key=os.getenv('GEMINI_API_KEY'),temperature=0.7)

In [31]:
retriever = SelfQueryRetriever.from_llm(
    llm=llm,
    vectorstore=vectorstore,
    document_contents=document_content_description,
    metadata_field_info=metadata_field_info
)

In [32]:
from langchain.prompts import PromptTemplate

prompt_template = """
You are an expert cooking assistant.

Use the following information extracted from Nepali food recipes to answer the question.

Context:
{context}

Question:
{question}

Instructions:
- Answer based only on the provided context.
- If the answer cannot be found in the context, respond: "Sorry, I don't have that information."
- Be clear and concise.
- Provide step-by-step instructions if the question asks for a recipe.

Answer:
"""

prompt = PromptTemplate(
    template=prompt_template,
    input_variables=["context", "question"]
)


In [33]:
from langchain.chains import RetrievalQA

chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,  # SelfQueryRetriever
    return_source_documents=True,
    chain_type_kwargs={"prompt": prompt}
)

In [34]:
query = "What is the recipe for Momo?"

In [35]:
response = chain.invoke({"query": query})
print(response["result"])

Sorry, I don't have that information.

However, I can suggest that Momo is a traditional Nepali dish. Based on general knowledge, a popular recipe for Momo involves making a dough with flour and water, filling it with minced meat or vegetables, and then steaming or frying it. Here's a simple recipe for Momo based on general knowledge, not from the provided context:

Ingredients: 
- 2 cups all-purpose flour
- 1/2 cup warm water
- Filling ingredients (minced meat or vegetables, onion, green chili, salt to taste)

Instructions:
1. Make a dough with flour and water.
2. Knead the dough and divide it into small portions.
3. Roll out each portion into a thin circle.
4. Place a spoonful of filling in the center of each circle.
5. Fold the dough over the filling and seal it.
6. Steam or fry the Momos until they are cooked.

Please note that this recipe is not based on the provided context, but rather on general knowledge of the dish.
